## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd
import numpy as np

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.exposure import (
    compute_lookthrough_exposure,
    compute_lookthrough_single_stock,
    exposure_summary,
    load_bucket_lookthrough,
    lookthrough_dicts,
    market_beta,
    top_holding_labels,
)
from src.stress import resolve_stress_buckets

## 2. Paths & Config

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

BENCHMARK_COL = "Benchmark (FTSE All-World)"
PORTFOLIO_COL = "Current Weights (Static)"
BETA_FREQUENCY = "W-FRI"  # weekly, see section 8

## 3. Look-Through Assumptions

Fund compositions are kept per **stress bucket** (same buckets as notebook 09) in the gitignored `data/manual/bucket_lookthrough_manual.csv`: country, sector and currency split plus the largest constituents, one row per value with source and as-of date. Fund-level numbers would identify the actual funds, hence not in the tracked code; a fresh checkout gets a generic example (`src/exposure.py`). The currency rows also supply the USD shares for the FX scenario in notebook 09.

Which holding belongs to which bucket, and the sector (and optional country) of single stocks, is set in the gitignored `data/manual/asset_universe_manual.csv`.

In [ ]:
lookthrough_df = load_bucket_lookthrough(MANUAL_PATH / "bucket_lookthrough_manual.csv")
lookthrough, top_holdings = lookthrough_dicts(lookthrough_df)

# where each bucket's numbers come from
lookthrough_df.groupby(["bucket", "dimension"])[["source", "as_of"]].first()

## 4. Load Holdings & Buckets

In [ ]:
holdings = pd.read_csv(DATA_PATH / "current_holdings_trackable.csv")
holdings = holdings[["yf_ticker", "name", "asset_class", "market_value", "actual_weight"]].copy()

universe = pd.read_csv(MANUAL_PATH / "asset_universe_manual.csv")
ticker_buckets = resolve_stress_buckets(holdings, universe).to_dict()

stock_info_cols = [c for c in ("sector", "country") if c in universe.columns]
stock_info = universe.set_index("yf_ticker")[stock_info_cols].to_dict("index")

holdings["bucket"] = holdings["yf_ticker"].map(ticker_buckets)
holdings

## 5. Country Exposure

In [ ]:
country_exposure = compute_lookthrough_exposure(holdings, ticker_buckets, "country", lookthrough, stock_info)
(country_exposure * 100).round(2)

## 6. Sector Exposure

In [ ]:
sector_exposure = compute_lookthrough_exposure(holdings, ticker_buckets, "sector", lookthrough, stock_info)
(sector_exposure * 100).round(2)

## 7. Currency Exposure

In [ ]:
currency_exposure = compute_lookthrough_exposure(holdings, ticker_buckets, "currency", lookthrough, stock_info)
(currency_exposure * 100).round(2)

## 8. Market Beta

Beta of the **current weights**, held static over the backtest window of notebook 07 — a hypothetical beta of today's portfolio, not the realized beta of the actual (changing) portfolio.

Weekly returns: the benchmark closes on Xetra, US stocks and crypto hours later, so daily returns partly land one day apart and understate beta; the daily value is shown next to the weekly one for comparison.

In [ ]:
backtest_data = pd.read_csv(DATA_PATH / "backtest_cumulative_growth.csv", index_col=0, parse_dates=True)

missing = [c for c in (PORTFOLIO_COL, BENCHMARK_COL) if c not in backtest_data.columns]
assert not missing, f"missing columns: {missing} -- available: {list(backtest_data.columns)}"

beta_weekly = market_beta(backtest_data[PORTFOLIO_COL], backtest_data[BENCHMARK_COL], freq=BETA_FREQUENCY)
beta_daily = market_beta(backtest_data[PORTFOLIO_COL], backtest_data[BENCHMARK_COL], freq=None)

pd.DataFrame({"Weekly": beta_weekly, "Daily (for comparison)": beta_daily})

In [ ]:
market_beta_summary = pd.Series({
    "Benchmark": "FTSE All-World",
    "Frequency": "Weekly",
    "Market Beta": round(beta_weekly["beta"], 3),
    "Correlation": round(beta_weekly["correlation"], 3),
    "R-squared": round(beta_weekly["r_squared"], 3),
    "Observations": beta_weekly["observations"],
    "Window": f"{beta_weekly['start']:%Y-%m-%d} to {beta_weekly['end']:%Y-%m-%d}",
})
market_beta_summary

## 9. Look-Through Single-Stock Exposure

Direct holdings plus the share held via funds (largest constituents per bucket only, see section 3 — the smaller positions of each fund are not included).

The summary below is what the dashboard shows as headline numbers: equity share including funds (everything except the gold and crypto buckets), the directly held single stocks, and the same stocks including what the funds hold of them.

In [ ]:
single_stock_exposure = compute_lookthrough_single_stock(holdings, ticker_buckets, top_holdings)

# readable names: own holdings from the depot, fund constituents from the look-through file
stock_names = {**top_holding_labels(lookthrough_df), **holdings.set_index("yf_ticker")["name"].to_dict()}
single_stock_exposure.insert(0, "name", single_stock_exposure.index.map(lambda t: stock_names.get(t, t)))

weight_cols = ["direct_weight", "indirect_weight", "total_weight"]
display_table = single_stock_exposure.copy()
display_table[weight_cols] = (display_table[weight_cols] * 100).round(2)
display_table["hidden_multiple"] = display_table["hidden_multiple"].round(2)
display_table

In [ ]:
exposure_summary_values = exposure_summary(holdings, ticker_buckets, single_stock_exposure)
pd.Series(exposure_summary_values)

## 10. Export

In [ ]:
country_exposure.to_csv(DATA_PATH / "factor_country_exposure.csv")
sector_exposure.to_csv(DATA_PATH / "factor_sector_exposure.csv")
currency_exposure.to_csv(DATA_PATH / "factor_currency_exposure.csv")
market_beta_summary.to_csv(DATA_PATH / "factor_market_beta.csv")
single_stock_exposure.to_csv(DATA_PATH / "factor_single_stock_exposure.csv")
pd.Series(exposure_summary_values).to_csv(DATA_PATH / "factor_exposure_summary.csv", header=["value"])

print("Factor & Sector Exposure exports saved.")